# 01 — Aggregate Metadata

Builds the spine table: every DragonForce studio album + track, pulled from MusicBrainz.
Everything downstream (lyrics, key-change annotation, whatever audio data we salvage)
joins to this table on `track_id`, which is derived from `recording_mbid` — **not** on
`album` + `track_title`. Titles drift between sources (feat. tags, `Pt. II` vs `Part 2`,
en-dashes, capitalization); MBIDs do not. See `PLAN.md`.

**This notebook is for exploring.** The dataset is built by `scripts/build_dataset.py`,
which is the reproducible path and the one whose output you should trust.

Run cells top to bottom the first time. After that, feel free to jump around — that's the whole point of notebooks.

In [4]:
import sys
sys.path.append("..")

import pandas as pd
from src.musicbrainz_client import get_release_groups, get_release_tracks

In [5]:
# Pull the album list. First run hits the MusicBrainz API and caches the raw
# JSON to data/raw/ (~1 request/sec, so give it a moment). Every run after that
# reads from disk — instant, and works with the wifi off. Pass refresh=True to
# force a fresh pull when you actually want updated data.
albums = get_release_groups()
albums_df = pd.DataFrame(albums)
albums_df[["title", "first-release-date", "primary-type"]].sort_values("first-release-date")

,title,first-release-date,primary-type
5,Valley of the Damned,2003-01-27,Album
8,Sonic Firestorm,2004-03-24,Album
7,Inhuman Rampage,2005-12-28,Album
4,Ultra Beatdown,2008-08-20,Album
2,Twilight Dementia,2010-09-08,Album
6,The Power Within,2012-04-11,Album
0,Maximum Overload,2014-08-08,Album
9,In the Line of Fire… Larger Than Live,2014-08-28,Album
12,Killer Elite,2016-04-13,Album
1,Reaching Into Infinity,2017-05-17,Album


In [ ]:
# The nine studio albums are hand-pinned in data/albums.json — one specific
# pressing each, chosen deliberately, with a note saying why. See PLAN.md.
#
# This cell used to filter the release-group list on "secondary-types" to drop
# live albums and compilations. That was fragile in two ways: the key is absent
# on some rows, so len(x) raised TypeError the moment MusicBrainz returned one,
# and the tagging itself is inconsistent enough that the track count could change
# between runs. With a pinned list neither problem is ours any more.
#
# What is still worth doing is the opposite direction: checking the pinned config
# against what MusicBrainz currently returns, so a new album (or a retitled one)
# shows up as a question rather than as a silently missing row.
import json

pinned = {a["title"] for a in json.load(open("../data/albums.json"))["albums"]}
returned = set(albums_df["title"])

new_to_musicbrainz = returned - pinned
missing_from_musicbrainz = pinned - returned

if new_to_musicbrainz:
    print("Not pinned in albums.json — a new release, or a non-studio one to ignore:")
    for title in sorted(new_to_musicbrainz):
        print("   ", title)
if missing_from_musicbrainz:
    print("Pinned but not returned by MusicBrainz — check the release-group MBID:")
    for title in sorted(missing_from_musicbrainz):
        print("   ", title)
if not new_to_musicbrainz and not missing_from_musicbrainz:
    print(f"albums.json matches MusicBrainz: {len(pinned)} albums, none new, none missing")


In [ ]:
# RESOLVED (PLAN.md Phase 1). get_release_tracks() needs a *release* MBID (a
# specific pressing), not a release-group MBID (the abstract album). Rather than
# automating "pick the canonical pressing" — a rabbit hole of deluxe editions,
# Japanese bonus tracks and promos that sort earlier than the real release — all
# nine were pinned by hand in data/albums.json with a note explaining each choice.
#
# The build lives in scripts/build_dataset.py. Explore here; build there.
#
# To pull one album's tracks for a look:
#
#     pinned = json.load(open("../data/albums.json"))["albums"]
#     tracks = get_release_tracks(pinned[0]["release_mbid"])
#     pd.DataFrame(tracks).head(20)


In [ ]:
# Don't save the spine from here. scripts/build_dataset.py writes
# data/processed/tracks.csv (and key_change_events.csv) behind a validation gate
# that refuses to write anything when a check fails. A CSV saved from a notebook
# cell bypasses all of it, and you cannot tell the two apart afterwards.
#
#     python scripts/build_dataset.py
